# 🏀 March Machine Learning Mania 2026# 

This notebook builds a machine learning pipeline to predict the probability that the lower TeamID beats the higher TeamID for every possible matchup in the 2026 NCAA Men’s and Women’s tournaments.

The evaluation metric for this competition is Brier Score.

#Pipeline Architecture

Data Loading
     →
Data Cleaning
     →
Regular Season Aggregation
     →
Season Team Features
     →
Feature Engineering (Team1 − Team2)
     →
Training Dataset Creation
     →
Time Series Cross Validation
     →
Model Training
    ├─ Logistic Regression (Baseline)
    └─ LightGBM (Main Model)
     →
Model Ensemble
     →
Probability Calibration
     →
Test Feature Generation
     →
Prediction for All Matchups
     →
Submission File

In [ ]:
#Data Loading
!kaggle competitions download -c march-machine-learning-mania-2026

In [ ]:
!unzip -o march-machine-learning-mania-2026.zip -d data

In [ ]:
import os
print(os.listdir("data"))

In [ ]:
#DOWNLOAD + LOAD DATA
import pandas as pd
import numpy as np
DATA_PATH = "data"

# MEN DATA
MTeams = pd.read_csv(f"{DATA_PATH}/MTeams.csv")
MSeasons = pd.read_csv(f"{DATA_PATH}/MSeasons.csv")
MRegularCompact = pd.read_csv(f"{DATA_PATH}/MRegularSeasonCompactResults.csv")
MRegularDetailed = pd.read_csv(f"{DATA_PATH}/MRegularSeasonDetailedResults.csv")
MNCAATourneyCompact = pd.read_csv(f"{DATA_PATH}/MNCAATourneyCompactResults.csv")
MNCAATourneyDetailed = pd.read_csv(f"{DATA_PATH}/MNCAATourneyDetailedResults.csv")
MNCAATourneySeeds = pd.read_csv(f"{DATA_PATH}/MNCAATourneySeeds.csv")
MMasseyOrdinals = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")

# WOMEN DATA
WTeams = pd.read_csv(f"{DATA_PATH}/WTeams.csv")
WSeasons = pd.read_csv(f"{DATA_PATH}/WSeasons.csv")
WRegularCompact = pd.read_csv(f"{DATA_PATH}/WRegularSeasonCompactResults.csv")
WRegularDetailed = pd.read_csv(f"{DATA_PATH}/WRegularSeasonDetailedResults.csv")
WNCAATourneyCompact = pd.read_csv(f"{DATA_PATH}/WNCAATourneyCompactResults.csv")
WNCAATourneyDetailed = pd.read_csv(f"{DATA_PATH}/WNCAATourneyDetailedResults.csv")
WNCAATourneySeeds = pd.read_csv(f"{DATA_PATH}/WNCAATourneySeeds.csv")

print("Data Loaded Successfully.")

In [ ]:
print(MRegularCompact.shape)
print(WRegularCompact.shape)

In [ ]:
# FILTER REGULAR SEASON DATA(No Leakage)

MRegularCompact = MRegularCompact[MRegularCompact["DayNum"] <= 132].copy()
MRegularDetailed = MRegularDetailed[MRegularDetailed["DayNum"] <= 132].copy()
WRegularCompact = WRegularCompact[WRegularCompact["DayNum"] <= 132].copy()
WRegularDetailed = WRegularDetailed[WRegularDetailed["DayNum"] <= 132].copy()
print("Regular season filtered correctly.")

In [ ]:
#Add Gender Flag

MRegularCompact["Gender"] = "M"
MRegularDetailed["Gender"] = "M"
MNCAATourneyCompact["Gender"] = "M"
MNCAATourneyDetailed["Gender"] = "M"

WRegularCompact["Gender"] = "W"
WRegularDetailed["Gender"] = "W"
WNCAATourneyCompact["Gender"] = "W"
WNCAATourneyDetailed["Gender"] = "W"

print("Gender flags added.")

# FEATURE ENGINE

Each game appears once (winner + loser).
We must convert into team-centric format.
So each game becomes:
* Row for winning team
* Row for losing team

Then aggregate.

In [ ]:
#Convert Detailed Results into Team-Level Rows (MEN)
# MEN TEAM-LEVEL GAME DATA

def build_team_game_df(detailed_df):
    win_df = detailed_df.copy()# Winner perspective
    win_df["TeamID"] = win_df["WTeamID"]
    win_df["OppTeamID"] = win_df["LTeamID"]
    win_df["PointsFor"] = win_df["WScore"]
    win_df["PointsAgainst"] = win_df["LScore"]
    win_df["FGM"] = win_df["WFGM"]
    win_df["FGA"] = win_df["WFGA"]
    win_df["FGM3"] = win_df["WFGM3"]
    win_df["FGA3"] = win_df["WFGA3"]
    win_df["FTM"] = win_df["WFTM"]
    win_df["FTA"] = win_df["WFTA"]
    win_df["OR"] = win_df["WOR"]
    win_df["DR"] = win_df["WDR"]
    win_df["Ast"] = win_df["WAst"]
    win_df["TO"] = win_df["WTO"]
    win_df["Stl"] = win_df["WStl"]
    win_df["Blk"] = win_df["WBlk"]
    win_df["PF"] = win_df["WPF"]
    win_df["Win"] = 1

    lose_df = detailed_df.copy()# Loser perspective
    lose_df["TeamID"] = lose_df["LTeamID"]
    lose_df["OppTeamID"] = lose_df["WTeamID"]
    lose_df["PointsFor"] = lose_df["LScore"]
    lose_df["PointsAgainst"] = lose_df["WScore"]
    lose_df["FGM"] = lose_df["LFGM"]
    lose_df["FGA"] = lose_df["LFGA"]
    lose_df["FGM3"] = lose_df["LFGM3"]
    lose_df["FGA3"] = lose_df["LFGA3"]
    lose_df["FTM"] = lose_df["LFTM"]
    lose_df["FTA"] = lose_df["LFTA"]
    lose_df["OR"] = lose_df["LOR"]
    lose_df["DR"] = lose_df["LDR"]
    lose_df["Ast"] = lose_df["LAst"]
    lose_df["TO"] = lose_df["LTO"]
    lose_df["Stl"] = lose_df["LStl"]
    lose_df["Blk"] = lose_df["LBlk"]
    lose_df["PF"] = lose_df["LPF"]
    lose_df["Win"] = 0
    combined = pd.concat([win_df, lose_df], ignore_index=True)
    keep_cols = ["Season","DayNum","TeamID","OppTeamID","Win","PointsFor","PointsAgainst","FGM","FGA","FGM3","FGA3","FTM","FTA","OR","DR","Ast","TO","Stl","Blk","PF"]
    return combined[keep_cols]

In [ ]:
# Build Men team-level data
M_team_games = build_team_game_df(MRegularDetailed)
print("Men team-game shape:", M_team_games.shape)

In [ ]:
# Build season level aggregate features(MEN)
def build_season_features(team_games_df):
    grouped = team_games_df.groupby(["Season","TeamID"])
    season_df = grouped.agg({
        "Win":"sum",
        "PointsFor":"sum",
        "PointsAgainst":"sum",
        "FGM":"sum",
        "FGA":"sum",
        "FGM3":"sum",
        "FTM":"sum",
        "FTA":"sum",
        "OR":"sum",
        "DR":"sum",
        "Ast":"sum",
        "TO":"sum",
        "Stl":"sum",
        "Blk":"sum",
        "PF":"sum",
        "DayNum":"count"
    }).rename(columns={"DayNum":"Games"})
    season_df["Losses"] = season_df["Games"] - season_df["Win"]
    season_df["WinPct"] = season_df["Win"] / season_df["Games"]
    season_df["AvgMargin"] = (season_df["PointsFor"] - season_df["PointsAgainst"]) / season_df["Games"]
    
    # Possessions estimate
    season_df["Possessions"] = (season_df["FGA"] - season_df["OR"] + season_df["TO"] + (0.475 * season_df["FTA"]))
    season_df["OffEff"] = season_df["PointsFor"] / season_df["Possessions"] * 100
    season_df["DefEff"] = season_df["PointsAgainst"] / season_df["Possessions"] * 100
    season_df["NetRating"] = season_df["OffEff"] - season_df["DefEff"]
    season_df = season_df.reset_index()
    return season_df
M_season_features = build_season_features(M_team_games)
print("Men season features shape:", M_season_features.shape)
M_season_features.head()

In [ ]:
print(M_season_features.describe())

In [ ]:
#Build women team game dataset
#detailed stats start 2010
W_team_games = build_team_game_df(WRegularDetailed)
print("Women team-game shape:", W_team_games.shape)

In [ ]:
#Build women season feature
W_season_features = build_season_features(W_team_games)
print("Women season features shape:", W_season_features.shape)

In [ ]:
W_season_features.head()

# ELO ENGINE

The Elo engine is a method used to rank and evaluate models in machine learning, particularly in the context of AI and language models.

In [ ]:
#ELO ENGINE
def compute_elo(team_games_df, base_elo=1500, k=20, home_adv=65, regression=0.25):
    df = team_games_df.sort_values(["Season","DayNum"]).copy()
    elo_dict = {}
    season_elos = []
    for idx, row in df.iterrows():
        season = row["Season"]
        team = row["TeamID"]
        opp = row["OppTeamID"]
        win = row["Win"]

        # Initialize if first appearance
        if (season, team) not in elo_dict:
            if (season-1, team) in elo_dict:
                prev_elo = elo_dict[(season-1, team)]
                elo_dict[(season, team)] = prev_elo * (1-regression) + base_elo * regression
            else:
                elo_dict[(season, team)] = base_elo
        if (season, opp) not in elo_dict:
            if (season-1, opp) in elo_dict:
                prev_elo = elo_dict[(season-1, opp)]
                elo_dict[(season, opp)] = prev_elo * (1-regression) + base_elo * regression
            else:
                elo_dict[(season, opp)] = base_elo
        team_elo = elo_dict[(season, team)]
        opp_elo = elo_dict[(season, opp)]

        # Expected score
        exp = 1 / (1 + 10 ** ((opp_elo - team_elo) / 400))

        # Margin multiplier
        margin = abs(row["PointsFor"] - row["PointsAgainst"])
        margin_mult = np.log(margin + 1) * (2.2 / ((team_elo - opp_elo) * 0.001 + 2.2))

        # Update
        new_elo = team_elo + k * margin_mult * (win - exp)
        elo_dict[(season, team)] = new_elo
        season_elos.append(team_elo)
    df["EloPre"] = season_elos
    return df

In [ ]:
#compute elo for men
M_team_games = compute_elo(M_team_games)
print("Men ELO Computer.")
M_team_games[["Season","TeamID","EloPre"]].head()

In [ ]:
#Extracting pre tournament elo of men. As we want elo before tournament
M_elo_season = (
    M_team_games
    .sort_values(["Season","DayNum"])
    .groupby(["Season","TeamID"])
    .tail(1)[["Season","TeamID","EloPre"]]
    .rename(columns={"EloPre":"EloRating"}))
print("Men Elo season shape:", M_elo_season.shape)

In [ ]:
#compute elo for women, same engine, separet pipeline
W_team_games = compute_elo(W_team_games)
print("Women Elo computed.")
W_team_games[["Season","TeamID", "EloPre"]].head()

In [ ]:
#Extracting pre tournament elo of women
W_elo_season = (
    W_team_games
    .sort_values(["Season","DayNum"])
    .groupby(["Season","TeamID"])
    .tail(1)[["Season","TeamID","EloPre"]]
    .rename(columns={"EloPre":"EloRating"}))
print("Women Elo season shape:", W_elo_season.shape)

In [ ]:
#merging elo into season features, enrich our feature tables

#MEN
M_season_features = M_season_features.merge(M_elo_season, on=["Season","TeamID"], how="left")
print("Men season features with Elo:", M_season_features.shape)

#WOMEN
W_season_features = W_season_features.merge(W_elo_season, on=["Season","TeamID"], how="left")
print("Women season features with Elo:", W_season_features.shape)

In [ ]:
print(M_season_features.columns)
print(W_season_features.columns)

For men, we merged twice 'EloRating_x', 'EloRating_y'(re-run twice). But for women its correct 'EloRating'.

In [ ]:
# FIX MEN ELO COLUMN. Drop duplicate column if exists

if "EloRating_y" in M_season_features.columns:
    M_season_features["EloRating"] = M_season_features["EloRating_y"]
    M_season_features = M_season_features.drop(columns=["EloRating_x", "EloRating_y"])
elif "EloRating_x" in M_season_features.columns:
    M_season_features = M_season_features.rename(columns={"EloRating_x": "EloRating"})
print("Men columns cleaned.")
print([col for col in M_season_features.columns if "Elo" in col])

In [ ]:
print(M_season_features["EloRating"].describe())
print(W_season_features["EloRating"].describe())

# Build Men training dataset

In [ ]:
# PREP MEN TOURNAMENT DATA
M_tourney = MNCAATourneyCompact.copy()

# Create lower and higher team IDs
M_tourney["Team1"] = M_tourney[["WTeamID","LTeamID"]].min(axis=1)
M_tourney["Team2"] = M_tourney[["WTeamID","LTeamID"]].max(axis=1)

# Target: 1 if lower ID team won
M_tourney["Target"] = (M_tourney["WTeamID"] == M_tourney["Team1"]).astype(int)
M_tourney = M_tourney[["Season","Team1","Team2","Target"]]
print("Men tournament games:", M_tourney.shape)
M_tourney.head()

In [ ]:
#MERGE MEN SEASON FEATURES

M_train = M_tourney.copy()

# Merge Team1 features
M_train = M_train.merge(M_season_features,left_on=["Season","Team1"],right_on=["Season","TeamID"],how="left",suffixes=("", "_T1"))

# Rename Team1 feature columns explicitly
for col in M_season_features.columns:
    if col not in ["Season","TeamID"]:
        M_train.rename(columns={col: f"{col}_T1"}, inplace=True)
M_train.drop(columns=["TeamID"], inplace=True)

# Merge Team2 features
M_train = M_train.merge(M_season_features,left_on=["Season","Team2"],right_on=["Season","TeamID"],how="left",suffixes=("", "_T2"))

# Rename Team2 feature columns explicitly
for col in M_season_features.columns:
    if col not in ["Season","TeamID"]:
        M_train.rename(columns={col: f"{col}_T2"}, inplace=True)
M_train.drop(columns=["TeamID"], inplace=True)

print("Men training shape:", M_train.shape)

In [ ]:
print(MNCAATourneySeeds.head())

In [ ]:
M_seeds = MNCAATourneySeeds.copy()
M_seeds["Seed"] = M_seeds["Seed"].str[1:3].astype(int)
M_seeds = M_seeds[["Season","TeamID","Seed"]]

In [ ]:
M_train = M_train.merge(
    M_seeds,
    left_on=["Season","Team1"],
    right_on=["Season","TeamID"],
    how="left"
)
M_train.rename(columns={"Seed":"Seed_T1"}, inplace=True)
M_train.drop(columns=["TeamID"], inplace=True)

In [ ]:
M_train = M_train.merge(
    M_seeds,
    left_on=["Season","Team2"],
    right_on=["Season","TeamID"],
    how="left"
)
M_train.rename(columns={"Seed":"Seed_T2"}, inplace=True)
M_train.drop(columns=["TeamID"], inplace=True)

In [ ]:
print([col for col in M_train.columns if "Seed" in col])

In [ ]:
# CREATING MEN DIFFERENCE FEATURES
#Raw features are not ideal, models learn best from differences

# List of core numeric features to difference
feature_cols = ["WinPct","AvgMargin","OffEff","DefEff","NetRating","Possessions","EloRating","Seed"]
for col in feature_cols:
    M_train[f"{col}_Diff"] = M_train[f"{col}_T1"] - M_train[f"{col}_T2"]

# Keep only relevant columns
diff_cols = [f"{col}_Diff" for col in feature_cols]
M_train_final = M_train[["Season","Team1","Team2","Target"] + diff_cols]
print("Men final training shape:", M_train_final.shape)
M_train_final.head()

All features diff for early seasons is NAN (Season=1985). But deatiled stats for Men start at 2003. Before 2003 no detailed features, merge produced NaN and also difference became NaNs. We restrict training data to season->2003.

In [ ]:
#Filter men training data (No Nan)
M_train_final = M_train_final[M_train_final["Season"]>=2003].copy()
print("Men training shape after filtering:", M_train_final.shape)
print("Any NaNs left?", M_train_final.isna().sum().sum())


# Building Women Training Dataset

In [ ]:
# STEP 4W-A: PREP WOMEN TOURNAMENT DATA

W_tourney = WNCAATourneyCompact.copy()
W_tourney["Team1"] = W_tourney[["WTeamID","LTeamID"]].min(axis=1)
W_tourney["Team2"] = W_tourney[["WTeamID","LTeamID"]].max(axis=1)
W_tourney["Target"] = (W_tourney["WTeamID"] == W_tourney["Team1"]).astype(int)
W_tourney = W_tourney[["Season","Team1","Team2","Target"]]
print("Women tournament shape:", W_tourney.shape)
W_tourney.head()

In [ ]:
# MERGE WOMEN SEASON FEATURES
W_train = W_tourney.copy()

# Merge Team1 features
W_train = W_train.merge(W_season_features,left_on=["Season","Team1"],right_on=["Season","TeamID"],how="left")
for col in W_season_features.columns:
    if col not in ["Season","TeamID"]:
        W_train.rename(columns={col: f"{col}_T1"}, inplace=True)
W_train.drop(columns=["TeamID"], inplace=True)

# Merge Team2 features
W_train = W_train.merge(W_season_features,left_on=["Season","Team2"],right_on=["Season","TeamID"],how="left")

for col in W_season_features.columns:
    if col not in ["Season","TeamID"]:
        W_train.rename(columns={col: f"{col}_T2"}, inplace=True)
W_train.drop(columns=["TeamID"], inplace=True)
print("Women training shape after merge:", W_train.shape)

In [ ]:
#feature differences
feature_cols = ["WinPct","AvgMargin","OffEff","DefEff","NetRating","Possessions","EloRating"]
for col in feature_cols:
    W_train[f"{col}_Diff"] = W_train[f"{col}_T1"] - W_train[f"{col}_T2"]
diff_cols = [f"{col}_Diff" for col in feature_cols]
W_train_final = W_train[["Season","Team1","Team2","Target"] + diff_cols]
print("Women final training shape:", W_train_final.shape)
W_train_final.head()

In [ ]:
#women stats start from 2010
W_train_final = W_train_final[W_train_final["Season"] >= 2010].copy()
print("Women training shape after filtering:", W_train_final.shape)
print("Any NaNs left?", W_train_final.isna().sum().sum())

# Time Series Cross Validation

In [ ]:
#Build Men CV Spilts
men_seasons = sorted(M_train_final["Season"].unique())
men_splits = []
for i in range(3, len(men_seasons)):  
    train_seasons = men_seasons[:i]
    val_season = men_seasons[i]
    men_splits.append((train_seasons, val_season))
print("Number of CV splits:", len(men_splits))
print("Example split:")
print("Train:", men_splits[0][0])
print("Validate:", men_splits[0][1])

In [ ]:
#Build Women CV Spilts
women_seasons = sorted(W_train_final["Season"].unique())
print("Women seasons:", women_seasons)
women_splits = []
for i in range(3, len(women_seasons)):
    train_seasons = women_seasons[:i]
    val_season = women_seasons[i]
    women_splits.append((train_seasons, val_season))
print("Number of CV splits:", len(women_splits))
print("Example split:")
print("Train:", women_splits[0][0])
print("Validate:", women_splits[0][1])

# Baseline Logistic regression(Brier Optimized)

In [ ]:
#Men Logistic CV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler
import numpy as np
men_features = [col for col in M_train_final.columns if "_Diff" in col]
men_log_oof = np.zeros(len(M_train_final))
men_brier_scores = []
for train_seasons, val_season in men_splits:
    train_df = M_train_final[M_train_final["Season"].isin(train_seasons)]
    val_df = M_train_final[M_train_final["Season"] == val_season]  
    val_idx = M_train_final["Season"] == val_season
    X_train = train_df[men_features]
    y_train = train_df["Target"]
    X_val = val_df[men_features]
    y_val = val_df["Target"]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    preds = model.predict_proba(X_val_scaled)[:,1]
    men_log_oof[val_idx] = preds
    brier = brier_score_loss(y_val, preds)
    men_brier_scores.append(brier)
print("Men CV Brier scores:")
print(men_brier_scores)
print("Mean Brier:", np.mean(men_brier_scores))

In [ ]:
#Women Logistic CV
women_features = [col for col in W_train_final.columns if "_Diff" in col]
women_log_oof = np.zeros(len(W_train_final))
women_brier_scores = []
for train_seasons, val_season in women_splits:
    train_df = W_train_final[W_train_final["Season"].isin(train_seasons)]
    val_df = W_train_final[W_train_final["Season"] == val_season]
    val_idx = W_train_final["Season"] == val_season
    X_train = train_df[women_features]
    y_train = train_df["Target"]
    X_val = val_df[women_features]
    y_val = val_df["Target"]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    preds = model.predict_proba(X_val_scaled)[:,1]
    women_log_oof[val_idx] = preds
    brier = brier_score_loss(y_val, preds)
    women_brier_scores.append(brier)
print("Women CV Brier scores:")
print(women_brier_scores)
print("Mean Brier:", np.mean(women_brier_scores))

# LightGBM 
Train lightgbm usinf time series cv

Store OOF prediction

Compute mean brier

In [ ]:
import lightgbm as lgb
from sklearn.metrics import brier_score_loss
import numpy as np
lgb_params = {"objective": "binary","metric": "binary_logloss","learning_rate": 0.03,"num_leaves": 31,"feature_fraction": 0.8,
              "bagging_fraction": 0.8,"bagging_freq": 5,"min_data_in_leaf": 30,"verbosity": -1}

In [ ]:
men_features = ["WinPct_Diff","AvgMargin_Diff","OffEff_Diff","DefEff_Diff","NetRating_Diff","Possessions_Diff","EloRating_Diff"]
men_lgb_oof = np.zeros(len(M_train_final))
men_lgb_scores = []
for train_seasons, val_season in men_splits:
    train_idx = M_train_final["Season"].isin(train_seasons)
    val_idx = M_train_final["Season"] == val_season
    X_train = M_train_final.loc[train_idx, men_features]
    y_train = M_train_final.loc[train_idx, "Target"]
    X_val = M_train_final.loc[val_idx, men_features]
    y_val = M_train_final.loc[val_idx, "Target"]
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)
    model = lgb.train(lgb_params,train_data,num_boost_round=1000,valid_sets=[val_data],callbacks=[lgb.early_stopping(100),lgb.log_evaluation(0)])
    preds = model.predict(X_val, num_iteration=model.best_iteration)
    men_lgb_oof[val_idx] = preds
    brier = brier_score_loss(y_val, preds)
    men_lgb_scores.append(brier)
print("MEN LGB Mean Brier:", np.mean(men_lgb_scores))

In [ ]:
women_features = ["WinPct_Diff","AvgMargin_Diff","OffEff_Diff","DefEff_Diff","NetRating_Diff","Possessions_Diff","EloRating_Diff"]
women_lgb_oof = np.zeros(len(W_train_final))
women_lgb_scores = []
for train_seasons, val_season in women_splits:
    train_idx = W_train_final["Season"].isin(train_seasons)
    val_idx = W_train_final["Season"] == val_season
    X_train = W_train_final.loc[train_idx, women_features]
    y_train = W_train_final.loc[train_idx, "Target"]
    X_val = W_train_final.loc[val_idx, women_features]
    y_val = W_train_final.loc[val_idx, "Target"]
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)
    model = lgb.train(lgb_params,train_data,num_boost_round=1000,valid_sets=[val_data],callbacks=[lgb.early_stopping(100),lgb.log_evaluation(0)])
    preds = model.predict(X_val, num_iteration=model.best_iteration)
    women_lgb_oof[val_idx] = preds
    brier = brier_score_loss(y_val, preds)
    women_lgb_scores.append(brier)
print("WOMEN LGB Mean Brier:", np.mean(women_lgb_scores))

#Logistic Baseline
* Men: 0.189709
* Women: 0.143312

#LightGBM
*  Men: 0.188356 (Improvement)
* Women: 0.147288 (Worse)

MEN
LightGBM improved from 0.1897 → 0.1884. That’s a real gain (~0.00135). Trees are capturing nonlinear structure. Men pipeline is behaving correctly

WOMEN
LightGBM worsened from 0.1433 → 0.1473. That’s not small noise — that’s meaningful degradation.
Why?
Women dataset: Smaller sample size, Lower variance, Stronger signal separation and More linear separability. Logistic already fits it very well. Trees are overfitting noise.

# XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import brier_score_loss
import numpy as np
xgb_params = {"objective": "binary:logistic","eval_metric": "logloss","learning_rate": 0.03,"max_depth": 4,"subsample": 0.8,
"colsample_bytree": 0.8,"min_child_weight": 5,"seed": 42,"verbosity": 0}

In [ ]:
men_features = ["WinPct_Diff", "AvgMargin_Diff", "OffEff_Diff","DefEff_Diff", "NetRating_Diff","Possessions_Diff", "EloRating_Diff"]
men_xgb_oof = np.zeros(len(M_train_final))
men_xgb_scores = []
for train_seasons, val_season in men_splits:
    train_idx = M_train_final["Season"].isin(train_seasons)
    val_idx = M_train_final["Season"] == val_season
    X_train = M_train_final.loc[train_idx, men_features]
    y_train = M_train_final.loc[train_idx, "Target"]
    X_val = M_train_final.loc[val_idx, men_features]
    y_val = M_train_final.loc[val_idx, "Target"]
    model = xgb.XGBClassifier( **xgb_params, n_estimators=2000)
    model.set_params(early_stopping_rounds=100)
    model.fit(X_train,y_train,eval_set=[(X_val, y_val)],verbose=False)
    preds = model.predict_proba(X_val)[:, 1]
    men_xgb_oof[val_idx] = preds
    brier = brier_score_loss(y_val, preds)
    men_xgb_scores.append(brier)
print("MEN XGB Mean Brier:", np.mean(men_xgb_scores))

In [ ]:
women_features = ["WinPct_Diff", "AvgMargin_Diff", "OffEff_Diff","DefEff_Diff", "NetRating_Diff","Possessions_Diff", "EloRating_Diff"]
women_xgb_oof = np.zeros(len(W_train_final))
women_xgb_scores = []
for train_seasons, val_season in women_splits:
    train_idx = W_train_final["Season"].isin(train_seasons)
    val_idx = W_train_final["Season"] == val_season
    X_train = W_train_final.loc[train_idx, women_features]
    y_train = W_train_final.loc[train_idx, "Target"]
    X_val = W_train_final.loc[val_idx, women_features]
    y_val = W_train_final.loc[val_idx, "Target"]
    model = xgb.XGBClassifier(**xgb_params,n_estimators=2000)
    model.fit(X_train,y_train,eval_set=[(X_val, y_val)],verbose=False)
    preds = model.predict_proba(X_val)[:, 1]
    women_xgb_oof[val_idx] = preds
    brier = brier_score_loss(y_val, preds)
    women_xgb_scores.append(brier)
print("WOMEN XGB Mean Brier:", np.mean(women_xgb_scores))

#MEN Model Ranking
* LightGBM (0.18836)
* XGBoost (0.18841)
* Logistic (0.18971)

Very close between LGB & XGB.

#WOMEN Model Ranking
* Logistic (0.14331)
* LightGBM (0.14729)
* XGBoost (0.17903)

Clear winner = Logistic.

In [ ]:
from sklearn.metrics import brier_score_loss
import numpy as np
y_true = M_train_final["Target"].values
men_ens_equal = (0.33 * men_log_oof +0.33 * men_lgb_oof +0.34 * men_xgb_oof)
print("MEN Equal Ensemble Brier:",brier_score_loss(y_true, men_ens_equal))

In [ ]:
best_score = 1
best_weights = None
for w1 in np.arange(0.0, 1.01, 0.05):
    for w2 in np.arange(0.0, 1.01 - w1, 0.05):
        w3 = 1 - w1 - w2
        preds = (w1 * men_log_oof + w2 * men_lgb_oof + w3 * men_xgb_oof)
        score = brier_score_loss(y_true, preds)
        if score < best_score:
            best_score = score
            best_weights = (w1, w2, w3)
print("BEST MEN ENSEMBLE Brier:", best_score)
print("Best Weights (Log, LGB, XGB):", best_weights)

Here Ensemble is worse Because your OOF arrays are wrong and ensemble Brier is 0.226, but individual models are 0.188.
That means the men_log_oof, men_lgb_oof, men_xgb_oof are not aligned properly with y_true.

Here XGB is useless and Logistic + LGB is best

So we must fix OOF alignment before moving forward.

In [ ]:
print(len(men_log_oof))
print(len(men_lgb_oof))
print(len(men_xgb_oof))
print(len(y_true))

In [ ]:
print(np.min(men_log_oof), np.max(men_log_oof))
print(np.min(men_lgb_oof), np.max(men_lgb_oof))
print(np.min(men_xgb_oof), np.max(men_xgb_oof))

In [ ]:
print("Logistic Brier:", brier_score_loss(y_true, men_log_oof))
print("LGB Brier:", brier_score_loss(y_true, men_lgb_oof))
print("XGB Brier:", brier_score_loss(y_true, men_xgb_oof))

In [ ]:
valid_mask = men_log_oof !=0

In [ ]:
print("Logistic OOF Brier:",brier_score_loss(y_true[valid_mask], men_log_oof[valid_mask]))
print("LGB OOF Brier:",brier_score_loss(y_true[valid_mask], men_lgb_oof[valid_mask]))
print("XGB OOF Brier:",brier_score_loss(y_true[valid_mask], men_xgb_oof[valid_mask]))

In [ ]:
men_ens = (0.45 * men_log_oof + 0.55 * men_lgb_oof)
print("Ensemble Brier:",brier_score_loss(y_true[valid_mask], men_ens[valid_mask]))

In [ ]:
#calibrate the ensemble oof prediction
from sklearn.isotonic import IsotonicRegression

# Use only valid rows
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(men_ens[valid_mask], y_true[valid_mask])
men_ens_cal = iso.predict(men_ens[valid_mask])
print("Calibrated Ensemble Brier:",brier_score_loss(y_true[valid_mask], men_ens_cal))

#Men
* Logistic = ~0.1897
* LGB = ~0.1886
* XGB = ~0.1886
* Ensemble = 0.1855
* Calibrated Ensemble = 0.18097

#Raw models were slightly miscalibrated

#Ensemble improved discrimination

#Isotonic fixed probability confidence

#Now probabilities are much better aligned with outcomes


In [ ]:
#For Women
from sklearn.metrics import brier_score_loss
y_true_w = W_train_final["Target"].values
valid_mask_w = women_log_oof != 0

In [ ]:
#check indivisual oof brier
print("Women Logistic OOF:",brier_score_loss(y_true_w[valid_mask_w],women_log_oof[valid_mask_w]))
print("Women LGB OOF:",brier_score_loss(y_true_w[valid_mask_w],women_lgb_oof[valid_mask_w]))
print("Women XGB OOF:",brier_score_loss(y_true_w[valid_mask_w],women_xgb_oof[valid_mask_w]))

In [ ]:
#weight search
best_score_w = 1
best_weights_w = None
for w1 in np.arange(0.0, 1.01, 0.05):
    for w2 in np.arange(0.0, 1.01 - w1, 0.05):
        w3 = 1 - w1 - w2
        preds = (w1 * women_log_oof + w2 * women_lgb_oof + w3 * women_xgb_oof)
        score = brier_score_loss(y_true_w[valid_mask_w],preds[valid_mask_w])
        if score < best_score_w:
            best_score_w = score
            best_weights_w = (w1, w2, w3)
print("BEST WOMEN ENSEMBLE Brier:", best_score_w)
print("Best Weights (Log, LGB, XGB):", best_weights_w)

In [ ]:
#build best ensemble
w1, w2, w3 = best_weights_w
women_ens = (w1 * women_log_oof + w2 * women_lgb_oof + w3 * women_xgb_oof)
print("Women Ensemble Brier:",brier_score_loss(y_true_w[valid_mask_w],women_ens[valid_mask_w]))

In [ ]:
#calibration
from sklearn.isotonic import IsotonicRegression
iso_w = IsotonicRegression(out_of_bounds="clip")
iso_w.fit(women_ens[valid_mask_w],y_true_w[valid_mask_w])
women_ens_cal = iso_w.predict( women_ens[valid_mask_w])
print("Calibrated Women Brier:",brier_score_loss(y_true_w[valid_mask_w],women_ens_cal))

#Women
* Logistic = 0.14337
* LGB = 0.14748
* XGB = 0.17923
* Ensemble = 0.14254
* Calibrated Ensemble = 0.13657



# Final Summary
* Men Calibrated Ensemble = 0.18097
* Women Calibrated Ensemble = 0.13657

In [ ]:
#Men Final Model
#define features
men_features = [col for col in M_train_final.columns if "_Diff" in col]

In [ ]:
#train logistics on all data
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
scaler_men = StandardScaler()
X_men_full = M_train_final[men_features]
y_men_full = M_train_final["Target"]
X_men_scaled = scaler_men.fit_transform(X_men_full)
log_full_men = LogisticRegression(max_iter=1000)
log_full_men.fit(X_men_scaled, y_men_full)

In [ ]:
# Train full LightGBM model
import lightgbm as lgb

# Prepare full training data
X_men_full = M_train_final[men_features]
y_men_full = M_train_final["Target"]

# LightGBM parameters (use same as CV)
lgb_params_men = {"objective": "binary","learning_rate": 0.05,"n_estimators": 1000,"num_leaves": 31,"subsample": 0.8,"colsample_bytree": 0.8,"random_state": 42}
lgb_full_men = lgb.LGBMClassifier(**lgb_params_men)
lgb_full_men.fit(X_men_full, y_men_full)

In [ ]:
#Women final Model
women_features = [col for col in W_train_final.columns if "_Diff" in col]

In [ ]:
#train logistics on all data
scaler_women = StandardScaler()
X_women_full = W_train_final[women_features]
y_women_full = W_train_final["Target"]
X_women_scaled = scaler_women.fit_transform(X_women_full)
log_full_women = LogisticRegression(max_iter=1000)
log_full_women.fit(X_women_scaled, y_women_full)

In [ ]:
# Train full LightGBM model
lgb_full_women = lgb.LGBMClassifier(**lgb_params)
lgb_full_women.fit(X_women_full, y_women_full)

In [ ]:
# LOAD SAMPLE SUBMISSION
sample_sub = pd.read_csv("data/SampleSubmissionStage1.csv")

# SPLIT ID COLUMN
ids = sample_sub["ID"].str.split("_", expand=True)
sample_sub["Season"] = ids[0].astype(int)
sample_sub["Team1"] = ids[1].astype(int)
sample_sub["Team2"] = ids[2].astype(int)

# SPLIT MEN AND WOMEN MATCHUPS
men_test = sample_sub[sample_sub["Team1"] < 3000].copy()
women_test = sample_sub[sample_sub["Team1"] >= 3000].copy()

#MEN SEEDS
men_test = men_test.merge(M_seeds, left_on=["Season","Team1"], right_on=["Season","TeamID"], how="left")
men_test.rename(columns={"Seed":"Seed_T1"}, inplace=True)
men_test.drop(columns=["TeamID"], inplace=True)
men_test = men_test.merge(M_seeds, left_on=["Season","Team2"], right_on=["Season","TeamID"], how="left")
men_test.rename(columns={"Seed":"Seed_T2"}, inplace=True)
men_test.drop(columns=["TeamID"], inplace=True)
men_test["Seed_T1"] = men_test["Seed_T1"].astype(str).str.extract(r'(\d+)').astype(float)
men_test["Seed_T2"] = men_test["Seed_T2"].astype(str).str.extract(r'(\d+)').astype(float)
men_test["Seed_T1"] = men_test["Seed_T1"].fillna(16)
men_test["Seed_T2"] = men_test["Seed_T2"].fillna(16)
men_test["Seed_Diff"] = men_test["Seed_T1"] - men_test["Seed_T2"]


# ---------- MEN TEST SET ----------
men_test = men_test.merge(M_season_features, left_on=["Season","Team1"], right_on=["Season","TeamID"], how="left")
for col in M_season_features.columns:
    if col not in ["Season","TeamID"]:
        men_test.rename(columns={col:f"{col}_T1"}, inplace=True)
men_test.drop(columns=["TeamID"], inplace=True)
men_test = men_test.merge(M_season_features, left_on=["Season","Team2"], right_on=["Season","TeamID"], how="left")
for col in M_season_features.columns:
    if col not in ["Season","TeamID"]:
        men_test.rename(columns={col:f"{col}_T2"}, inplace=True)
men_test.drop(columns=["TeamID"], inplace=True)

# CREATE SAME FEATURES AS TRAINING
men_diff_cols = ["WinPct_Diff","AvgMargin_Diff","OffEff_Diff","DefEff_Diff","NetRating_Diff","EloRating_Diff","Possessions_Diff","Seed_Diff"]
for col in men_diff_cols:
    base = col.replace("_Diff","")
    men_test[col] = men_test[f"{base}_T1"] - men_test[f"{base}_T2"]
X_men_test = men_test[lgb_full_men.feature_name_].fillna(0)
men_test["Pred"] = lgb_full_men.predict_proba(X_men_test)[:,1]

# ---------- WOMEN TEST SET ----------
women_test = women_test.merge(W_season_features, left_on=["Season","Team1"], right_on=["Season","TeamID"], how="left")
for col in W_season_features.columns:
    if col not in ["Season","TeamID"]:
        women_test.rename(columns={col:f"{col}_T1"}, inplace=True)
women_test.drop(columns=["TeamID"], inplace=True)
women_test = women_test.merge(W_season_features, left_on=["Season","Team2"], right_on=["Season","TeamID"], how="left")
for col in W_season_features.columns:
    if col not in ["Season","TeamID"]:
        women_test.rename(columns={col:f"{col}_T2"}, inplace=True)
women_test.drop(columns=["TeamID"], inplace=True)

# CREATE SAME FEATURES AS TRAINING
women_diff_cols = ["WinPct_Diff","AvgMargin_Diff","OffEff_Diff","DefEff_Diff","NetRating_Diff","EloRating_Diff","Possessions_Diff"]
for col in women_diff_cols:
    base = col.replace("_Diff","")
    women_test[col] = women_test[f"{base}_T1"] - women_test[f"{base}_T2"]
X_women_test = women_test[lgb_full_women.feature_name_].fillna(0)
women_test["Pred"] = lgb_full_women.predict_proba(X_women_test)[:,1]

# Create Submission

In [ ]:
#---------- COMBINE ----------
men_sub = men_test[["ID","Pred"]]
women_sub = women_test[["ID","Pred"]]
submission = pd.concat([men_sub, women_sub])
submission = submission.sort_values("ID")
print("Submission shape:", submission.shape)
submission.to_csv("submission.csv", index=False)

In [ ]:
print(submission.shape)
print(submission.head())
print(submission.isnull().sum())

In [ ]:
#Download
from IPython.display import FileLink
FileLink("submission.csv")

# Distribution of Predicted Probablities

In [ ]:
#MEN
import matplotlib.pyplot as plt
plt.figure(figsize=(6,3))
plt.hist(men_test["Pred"], bins=50)
plt.title("Men Prediction Probability Distribution")
plt.xlabel("Predicted Win Probability")
plt.ylabel("Count")
plt.show()

In [ ]:
#WOMEN
plt.figure(figsize=(6,3))
plt.hist(women_test["Pred"], bins=50)
plt.title("Women Prediction Probability Distribution")
plt.xlabel("Predicted Win Probability")
plt.ylabel("Count")
plt.show()

# Feature Importance

LightGBM can show which features matter most

In [ ]:
#MEN
import pandas as pd
importance = pd.DataFrame({"Feature": lgb_full_men.feature_name_,"Importance": lgb_full_men.feature_importances_}).sort_values("Importance", ascending=False)
plt.figure(figsize=(6,3))
plt.barh(importance["Feature"], importance["Importance"])
plt.title("Men Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
#WOMEN
importance_w = pd.DataFrame({"Feature": lgb_full_women.feature_name_,"Importance": lgb_full_women.feature_importances_}).sort_values("Importance", ascending=False)
plt.figure(figsize=(6,3))
plt.barh(importance_w["Feature"], importance_w["Importance"])
plt.title("Women Feature Importance")
plt.gca().invert_yaxis()
plt.show()

# Future Work

* Advanced Elo Rating System: Implement a dynamic Elo rating that updates after every game to better capture team strength throughout the season.

* Advanced Ensemble Models: Combine additional models such as XGBoost, CatBoost, or Neural Networks for stronger predictive performance.

* Feature Interaction Engineering: Create interaction features between offensive and defensive metrics to better capture matchup dynamics.


Good luck!!!
